In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings 
warnings.filterwarnings('ignore')

In [4]:
df=pd.read_csv("Reviews.csv")
df.head(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...


In [5]:
df = df.sample(n=50000, random_state=42).reset_index(drop=True)

In [7]:
df.shape

(50000, 10)

In [8]:
df = df[['Score', 'Summary', 'Text']]

In [9]:
print(df.shape)
print(df.head(3))
print("\nScore distribution:")
print(df['Score'].value_counts().sort_index())

(50000, 3)
   Score                                       Summary  \
0      5  Crunchy & Good Gluten-Free Sandwich Cookies!   
1      5                            great kitty treats   
2      3                                  COFFEE TASTE   

                                                Text  
0  Having tried a couple of other brands of glute...  
1  My cat loves these treats. If ever I can't fin...  
2  A little less than I expected.  It tends to ha...  

Score distribution:
Score
1     4528
2     2576
3     3791
4     7008
5    32097
Name: count, dtype: int64


In [10]:
# Combine Summary + Text into one column
df['content'] = df['Summary'] + ' ' + df['Text']

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Score    50000 non-null  int64 
 1   Summary  49999 non-null  object
 2   Text     50000 non-null  object
 3   content  49999 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


In [15]:
def clean_text(text):
    # Handle missing values
    if not isinstance(text, str):
        return ''
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # 4. Keep only letters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 5. Tokenize
    tokens = text.split()
    
    # 6. Remove stopwords + lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    
    return ' '.join(tokens)

In [16]:
df['cleaned'] = df['content'].apply(clean_text)

print("✅ Done!")
print(f"Null values in cleaned: {df['cleaned'].isnull().sum()}")
print("\nSample cleaned review:")
print(df['cleaned'][5])

✅ Done!
Null values in cleaned: 0

Sample cleaned review:
lifesaver pineapple flavor please add pineapple flavor package lifesaver fact sell pineapple flavor


In [17]:
# Compare average length before and after cleaning
df['raw_length'] = df['content'].apply(lambda x: len(str(x).split()))
df['clean_length'] = df['cleaned'].apply(lambda x: len(x.split()))

print("=== Length Comparison ===")
print(f"Avg words BEFORE cleaning: {df['raw_length'].mean():.1f}")
print(f"Avg words AFTER cleaning:  {df['clean_length'].mean():.1f}")
print(f"Vocabulary reduced by:     {(1 - df['clean_length'].mean()/df['raw_length'].mean())*100:.1f}%")

# Save so we don't redo this next session
df.to_csv('reviews_cleaned.csv', index=False)
print("\n✅ Saved to reviews_cleaned.csv")

=== Length Comparison ===
Avg words BEFORE cleaning: 84.2
Avg words AFTER cleaning:  42.8
Vocabulary reduced by:     49.1%

✅ Saved to reviews_cleaned.csv


In [18]:
df

,Score,Summary,Text,content,cleaned,raw_length,clean_length
0,5,Crunchy & Good Gluten-Free Sandwich Cookies!,Having tried a couple of other brands of glute...,Crunchy & Good Gluten-Free Sandwich Cookies! H...,crunchy good glutenfree sandwich cooky tried c...,90,44
1,5,great kitty treats,My cat loves these treats. If ever I can't fin...,great kitty treats My cat loves these treats. ...,great kitty treat cat love treat ever cant fin...,102,50
2,3,COFFEE TASTE,A little less than I expected. It tends to ha...,COFFEE TASTE A little less than I expected. I...,coffee taste little less expected tends muddy ...,30,13
3,2,So the Mini-Wheats were too big?,"First there was Frosted Mini-Wheats, in origin...",So the Mini-Wheats were too big? First there w...,miniwheats big first frosted miniwheats origin...,300,143
4,5,Great Taste . . .,and I want to congratulate the graphic artist ...,Great Taste . . . and I want to congratulate t...,great taste want congratulate graphic artist p...,127,60
...,...,...,...,...,...,...,...
49995,5,This is a staple at my house,"I love this tea! I'd underestimated that, how...",This is a staple at my house I love this tea! ...,staple house love tea id underestimated howeve...,363,190
49996,3,Not as light as expected,I like a light strength of coffee and this one...,Not as light as expected I like a light streng...,light expected like light strength coffee one ...,58,26
49997,5,Natural Sugar,This is the best sugar out there...I carry a l...,Natural Sugar This is the best sugar out there...,natural sugar best sugar therei carry little b...,48,29
49998,5,FLAVORLESS Coconut Oil,I love coconut oil for it's high heat toleranc...,FLAVORLESS Coconut Oil I love coconut oil for ...,flavorless coconut oil love coconut oil high h...,42,26


NameError: name 'text' is not defined